In [1]:
import pandas as pd

In [6]:
df_cpi = pd.read_csv(r'C:\Users\MSI PULSE GL66\Documents\UNISEL\Degree\Sem 7\Data Analytic\01_data_cleaning\cpi_2d_state.csv')
df_hh = pd.read_csv(r'C:\Users\MSI PULSE GL66\Documents\UNISEL\Degree\Sem 7\Data Analytic\01_data_cleaning\hh_income_state.csv')
df_hies = pd.read_csv(r'C:\Users\MSI PULSE GL66\Documents\UNISEL\Degree\Sem 7\Data Analytic\01_data_cleaning\hies_state.csv')

# Diagnostic Check: Inspect data types and missing values
print('CPI Missing Values')
print(df_cpi.isnull().sum())

print('Income Missing Values')
print(df_hh.isnull().sum())

print('HIES Missing Values')
print(df_hies.isnull().sum())


CPI Missing Values
state       0
date        0
division    0
index       0
dtype: int64
Income Missing Values
state             0
date              0
income_mean       0
income_median    11
dtype: int64
HIES Missing Values
date                0
state               0
income_mean         0
income_median       0
expenditure_mean    0
gini                0
poverty             0
dtype: int64


In [10]:
# Convert 'date' from text (object) to proper datetime format, then extract the year

for df in [df_cpi, df_hh, df_hies]:
    df['date'] = pd.to_datetime(df['date'])
    df['year'] = df['date'].dt.year

print("CPI Years:", df_cpi['year'].min(), "to", df_cpi['year'].max())
print('Income Year:', df_hh['year'].unique())

CPI Years: 2010 to 2026
Income Year: [1970 1974 1976 1979 1984 1987 1989 1992 1995 1997 1999 2002 2004 2007
 2009 2012 2014 2016 2019 2020 2022 2024]


In [15]:
# Filter historical income from 2010 onwards and select clean columns
fact_income_history = (df_hh[df_hh['year'] >= 2010][['state','date','income_mean','income_median']].copy())

#verify the nan values remains
print("Remaining Null values \n", fact_income_history.isnull().sum())

Remaining Null values 
 state            0
date             0
income_mean      0
income_median    0
dtype: int64


In [ ]:
# Create a dictionary mapping official UN COICOP 2-digit codes to readable category names
dm = {
   'overall': 'Overall CPI',
    '01': 'Food & Beverages',
    '02': 'Alcohol & Tobacco',
    '03': 'Clothing & Footwear',
    '04': 'Housing, Utilities & Fuel',
    '05': 'Furnishings & Household Maintenance',
    '06': 'Health',
    '07': 'Transport',
    '08': 'Information & Communication',
    '09': 'Recreation, Sport & Culture',
    '10': 'Education',
    '11': 'Restaurants & Accommodation',
    '12': 'Insurance & Financial Services',
    '13': 'Personal Care & Miscellaneous'

}

In [17]:
# Map codes to readable names
df_cpi['category'] = df_cpi['division'].map(dm)

In [28]:
# Group by state, year, division, and category to calculate Annual Average CPI
fact_cpi_annual = (df_cpi.groupby(['state', 'year', 'division', 'category'], as_index=False)['index'].mean().round(2).rename(columns={"index": 'cpi_index'}))

print(fact_cpi_annual.tail())

               state  year division                        category  cpi_index
3803  W.P. Putrajaya  2026       10                       Education     132.90
3804  W.P. Putrajaya  2026       11     Restaurants & Accommodation     180.60
3805  W.P. Putrajaya  2026       12  Insurance & Financial Services     120.29
3806  W.P. Putrajaya  2026       13   Personal Care & Miscellaneous     141.73
3807  W.P. Putrajaya  2026  overall                     Overall CPI     143.70


In [29]:
# Select clean columns for the 2022 & 2024 HIES Fact Table
fact_hies = (df_hies[['state', 'year', 'income_mean', 'income_median', 'expenditure_mean', 'gini', 'poverty']].copy())

In [30]:
rm = {
    'Selangor': 'Central', 'W.P. Kuala Lumpur': 'Central', 'W.P. Putrajaya': 'Central', 'Negeri Sembilan': 'Central',
    'Pulau Pinang': 'Northern', 'Perak': 'Northern', 'Kedah': 'Northern', 'Perlis': 'Northern',
    'Johor': 'Southern', 'Melaka': 'Southern',
    'Pahang': 'East Coast', 'Terengganu': 'East Coast', 'Kelantan': 'East Coast',
    'Sabah': 'East Malaysia', 'Sarawak': 'East Malaysia', 'W.P. Labuan': 'East Malaysia'
}

In [31]:
dim_states = pd.DataFrame({'state': sorted(df_hies['state'].unique())})
dim_states['region'] = dim_states['state'].map(rm)

print(dim_states)

                state         region
0               Johor       Southern
1               Kedah       Northern
2            Kelantan     East Coast
3              Melaka       Southern
4     Negeri Sembilan        Central
5              Pahang     East Coast
6               Perak       Northern
7              Perlis       Northern
8        Pulau Pinang       Northern
9               Sabah  East Malaysia
10            Sarawak  East Malaysia
11           Selangor        Central
12         Terengganu     East Coast
13  W.P. Kuala Lumpur        Central
14        W.P. Labuan  East Malaysia
15     W.P. Putrajaya        Central


In [32]:
fact_income_history.to_csv("clean_fact_income_history.csv", index=False)
fact_cpi_annual.to_csv("clean_fact_cpi_annual.csv", index=False)
fact_hies.to_csv("clean_fact_hies.csv", index=False)
dim_states.to_csv("clean_dim_states.csv", index=False)

print("SUCCESS")

SUCCESS
